# Learning objectives

- Reproduce portal file uploads via the SDK.
- Create a participant-scoped vector store and attach File Search.
- Verify grounded responses.

In [ ]:
# Import local configuration helpers, Foundry SDK types, and file-system support.
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from workshop_core.config import WorkshopConfig
from workshop_core.naming import build_resource_name, sdk_participant_id

# Load the participant configuration and reserve the independent SDK namespace.
load_dotenv()
config = WorkshopConfig.load()
sdk_participant = sdk_participant_id(config.participant_id)

# Build stable names for the vector store and the grounded agent version.
vector_store_name = build_resource_name(
    sdk_participant,
    "grounding",
    "vector-store",
)
agent_name = build_resource_name(sdk_participant, "grounding", "agent")
print(f"Portal participant ID: {config.participant_id}")
print(f"SDK participant ID: {sdk_participant}")
print(f"Vector store name: {vector_store_name}")
print(f"SDK agent name: {agent_name}")

## Vector store and file search

Authenticate and create project client

In [ ]:
# Authenticate with the Azure identity established by az login and connect to Foundry.
credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential,
)
openai = project.get_openai_client()

# Check for a resource with the exact participant-scoped vector-store name.
existing = [
    store
    for store in openai.vector_stores.list()
    if getattr(store, "name", None) == vector_store_name
]
if existing:
    raise RuntimeError(
        f"{vector_store_name} already exists. Choose a unique PARTICIPANT_ID."
    )


Upload the four support Markdown files from `assets/support-knowledge/`, create the vector store, attach files, and poll until indexing completes.

In [ ]:
# Import the timer used to place a bounded limit on file indexing.
from time import time

# Define and validate the four files that form this lab's knowledge source.
expected_names = [
    "product-overview.md",
    "support-policy.md",
    "troubleshooting.md",
    "warranty-returns.md",
]
ASSET_DIR = Path("../../../assets/support-knowledge")
files = sorted(ASSET_DIR.glob("*.md"))
found_names = [path.name for path in files]
if set(found_names) != set(expected_names):
    raise RuntimeError(
        f"Expected exactly the four support assets: {expected_names}; found: {found_names}"
    )
print("Files to upload:", found_names)

# Create an empty Foundry vector store, then give indexing five minutes to complete.
timeout_seconds = 300
deadline = time() + timeout_seconds
vector_store = openai.vector_stores.create(name=vector_store_name)
print(f"Created vector store ID: {vector_store.id}")

# Upload each Markdown file to the vector store and wait for its indexing result.
uploaded_records = []
for path in files:
    with path.open("rb") as file_handle:
        file_record = openai.vector_stores.files.upload_and_poll(
            vector_store_id=vector_store.id,
            file=file_handle,
        )
    print(
        f"Uploaded {path.name} -> {getattr(file_record, 'id', None)} "
        f"status={getattr(file_record, 'status', None)}"
    )
    uploaded_records.append(file_record)
    if time() > deadline:
        raise RuntimeError(
            f"Indexing timed out for vector store {vector_store.id}. "
            "See docs/prerequisites/infrastructure.md."
        )

# Record the completed file IDs so they can be inspected during the lab.
for record in uploaded_records:
    print(
        f"Final file: {getattr(record, 'id', None)} "
        f"status={getattr(record, 'status', None)}"
    )
print(f"\nVector store ID: {vector_store.id}")
print("Final vector-store ingestion status: completed")

# Verify Grounded Response

Create the grounded agent with the same support-assistant behavior as Lab 1, then verify the E17 response includes the required troubleshooting terms.   
Inspect the response to confirm that it is grounded in the uploaded support knowledge.

In [ ]:
# Reuse the baseline support behavior and ask a question covered by the uploaded corpus.
instructions = (
    "You are the Contoso SupportHub X1 support assistant. "
    "Answer only from connected sources when they are available. "
    "If a source does not contain the answer, say so. "
    "Never invent customer or private information."
)
prompt = "What should I do when error E17 appears twice?"

# Create an agent version with File Search attached to this vector store.
grounded_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=config.model_deployment_name,
        instructions=instructions,
        tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    ),
)

# Create a managed conversation and send the question to the grounded agent.
conversation = openai.conversations.create()
grounded_response = openai.responses.create(
    conversation=conversation.id,
    input=prompt,
    extra_body={
        "agent_reference": {
            "name": grounded_agent.name,
            "type": "agent_reference",
        },
    },
)

# Check for the required troubleshooting concepts, then inspect the grounded response.
response_text = getattr(grounded_response, "output_text", "") or ""
print("Grounded response:\n", response_text)

# Challenge

Ask whether a device can be returned after 21 days.   
Inspect the response to confirm that it is grounded in the uploaded support knowledge, then repeat after removing File Search from a new agent version and compare the answers.

# Example solution

This solution identifies `warranty-returns.md` as the relevant workshop source, then compares the grounded response with a new agent version that has no File Search tool attached.

In [ ]:
## Code cell for the challenge, write your code below this line.

# Example solution - spoilers to the challenge

In [ ]:
# Ask a return-policy question that is answered by the workshop warranty document.
return_prompt = "Can I return a SupportHub X1 after 21 days?"

# Run the question through the agent version that has File Search enabled.
grounded_return_response = openai.responses.create(
    conversation=conversation.id,
    input=return_prompt,
    extra_body={
        "agent_reference": {
            "name": grounded_agent.name,
            "type": "agent_reference",
        },
    },
)
grounded_return_text = getattr(grounded_return_response, "output_text", "") or ""
assert "30 calendar days" in grounded_return_text.lower()
print(f"Grounded response:\n{grounded_return_text}")

# Create a comparison version with the same instructions but no File Search tool.
ungrounded_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=config.model_deployment_name,
        instructions=instructions,
    ),
)
ungrounded_return_response = openai.responses.create(
    input=return_prompt,
    extra_body={
        "agent_reference": {
            "name": ungrounded_agent.name,
            "type": "agent_reference",
        },
    },
)
print("\nUngrounded response:\n", ungrounded_return_response.output_text)